# Analyse SimStock — exemple d'analyse d'un run

**Run :** `{SAVE_NAME}` (à configurer ci-dessous) · référence 2024
**Objectif :** sanity check, calibration, distribution, substitutions, cas d'usage, optimisation du seuil, verdict Go/No-Go.

> Les cellules s'exécutent depuis le dossier `notebooks/` ou depuis la racine du repo.


## 0 · Imports & configuration

In [ ]:
import sys, os
from pathlib import Path

# Résoudre la racine du repo
_nb_dir = Path.cwd()
PROJECT_ROOT = _nb_dir.parent if (_nb_dir.parent / 'euronext_simstock').exists() else _nb_dir
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

# ── Configuration ──────────────────────────────────────────────────────
SAVE_NAME = "your_run_name"  # remplacer par ton save_name, ex : "euronext_simstock_v1"
# ────────────────────────────────────────────────────────────────────────

import sys, warnings
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from glob import glob
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import roc_curve, auc

from euronext_simstock.similarity.substitution import SubstitutionEngine
from euronext_simstock.config import SubstitutionConfig

FIGURES_DIR = Path("notebooks/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
sns.set_theme(style="whitegrid")

print("Imports OK")


## 0.1 · Chargement des artefacts

In [ ]:
EMB_DIR = PROJECT_ROOT / 'artifacts' / 'embeddings'

# Chargement automatique des artefacts du run
daily_path  = next(EMB_DIR.glob(f"{SAVE_NAME}_daily.parquet"))
stock_path  = next(EMB_DIR.glob(f"{SAVE_NAME}_stock.parquet"))
engine_path = next(EMB_DIR.glob(f"{SAVE_NAME}_engine.npz"))
simmat_path = next(EMB_DIR.glob(f"{SAVE_NAME}_cosine_simmat.parquet"))
diag_path   = next(EMB_DIR.glob(f"{SAVE_NAME}_similarity_diagnostics.parquet"))
table_path  = next(EMB_DIR.glob(f"{SAVE_NAME}_substitution_table.parquet"))
long_path   = next(EMB_DIR.glob(f"{SAVE_NAME}_substitution_long.parquet"))

emb_daily = pd.read_parquet(daily_path)
emb_stock = pd.read_parquet(stock_path)

CFG = SubstitutionConfig(similarity_threshold=0.85)
engine = SubstitutionEngine.load(engine_path, cfg=CFG)

# Matrices & vecteurs de référence
sim_mat    = engine.similarity_matrix          # np.ndarray (N, N)
tickers    = engine.tickers                    # list[str]
N          = len(tickers)
sector_arr = np.array([engine.sector_lookup.get(t, "Unknown") for t in tickers])
unique_sectors = sorted(set(sector_arr))

e_cols = [c for c in emb_stock.columns if c.startswith("e_")]
# Aligner sur l'ordre engine
emb_matrix = emb_stock.set_index("ticker").loc[tickers, e_cols].values.astype(np.float32)

print(f"Embeddings daily : {emb_daily.shape}")
print(f"Embeddings stock : {emb_matrix.shape}  ({len(e_cols)} dims)")
print(f"Tickers dans engine : {N}")
print(f"Secteurs : {len(unique_sectors)}  {unique_sectors}")


---
## Section 1 · Sanity check des embeddings

On vérifie que les représentations apprises sont numériquement saines et qu'elles séparent les secteurs.
**Note :** pendant l'entraînement, la normalisation L2 s'applique uniquement à l'intérieur du triplet loss (`anchor_n = F.normalize(anchor_emb)`).
Les embeddings CLS extraits via `extract_embeddings()` sont **bruts** (non normalisés) — les normes ne seront donc pas toutes ~1.
La similarité cosine dans l'engine normalise en interne avant de calculer le produit scalaire.


In [ ]:
# 1.1 · Distribution des normes L2
norms = np.linalg.norm(emb_matrix, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(norms, bins=60, color="steelblue", edgecolor="white", alpha=0.85)
axes[0].axvline(norms.mean(), color="red", linestyle="--", lw=2, label=f"Moyenne = {norms.mean():.2f}")
axes[0].axvline(norms.median() if hasattr(norms, 'median') else np.median(norms),
                color="orange", linestyle=":", lw=2, label=f"Médiane = {np.median(norms):.2f}")
axes[0].set_xlabel("Norme L2")
axes[0].set_ylabel("Nombre de tickers")
axes[0].set_title("Distribution des normes L2 (stock-level)")
axes[0].legend()

# QQ-like boxplot par secteur
sector_norms = {s: norms[sector_arr == s] for s in unique_sectors}
axes[1].boxplot(
    [sector_norms[s] for s in unique_sectors],
    labels=[s[:12] for s in unique_sectors],
    vert=False, patch_artist=True,
)
axes[1].set_xlabel("Norme L2")
axes[1].set_title("Normes L2 par secteur")
plt.setp(axes[1].get_yticklabels(), fontsize=8)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "S1_l2_norms.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"Normes L2 — min: {norms.min():.3f}  mean: {norms.mean():.3f}  "
      f"max: {norms.max():.3f}  std: {norms.std():.3f}")
print("Les embeddings ne sont PAS normalisés en sortie du modèle (normalisation interne au triplet loss uniquement).")


### 1.2 · PCA 2D colorée par secteur

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(emb_matrix)

palette = sns.color_palette("tab20", len(unique_sectors))
sec_color = {s: palette[i] for i, s in enumerate(unique_sectors)}

fig, ax = plt.subplots(figsize=(12, 8))
for s in unique_sectors:
    mask = sector_arr == s
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=[sec_color[s]], label=s, s=18, alpha=0.75, linewidths=0)

var_total = pca.explained_variance_ratio_.sum() * 100
ax.set_title(f"PCA 2D — embeddings stock-level (variance expliquée : {var_total:.1f} %)")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f} %)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f} %)")
ax.legend(loc="upper right", fontsize=8, markerscale=2, framealpha=0.8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "S1_pca_2d.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"Variance expliquée PC1+PC2 : {var_total:.1f} %")


### 1.3 · t-SNE 2D colorée par secteur (perplexity=50, n_iter=1000)

In [ ]:
print("Calcul t-SNE en cours (1-3 min)…")
tsne = TSNE(n_components=2, perplexity=50, max_iter=1000, random_state=42, n_jobs=-1)
X_tsne = tsne.fit_transform(emb_matrix)
print(f"KL divergence finale : {tsne.kl_divergence_:.4f}")

fig, ax = plt.subplots(figsize=(12, 8))
for s in unique_sectors:
    mask = sector_arr == s
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
               c=[sec_color[s]], label=s, s=18, alpha=0.75, linewidths=0)

ax.set_title("t-SNE 2D — embeddings stock-level (perplexity=50)")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.legend(loc="upper right", fontsize=8, markerscale=2, framealpha=0.8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "S1_tsne_2d.png", dpi=120, bbox_inches="tight")
plt.show()


### 1.4 · Conclusion : clusters sectoriels

In [ ]:
# Quantification de la séparation sectorielle via le ratio intra/inter distances
from sklearn.metrics import pairwise_distances

# Centroïdes par secteur (sur embeddings normalisés)
norms_col = np.linalg.norm(emb_matrix, axis=1, keepdims=True)
emb_normed = emb_matrix / np.where(norms_col == 0, 1, norms_col)

centroids = {}
for s in unique_sectors:
    mask = sector_arr == s
    if mask.sum() > 0:
        centroids[s] = emb_normed[mask].mean(axis=0)

# Silhouette approximative : distance intra-secteur vs inter-secteur
intra_dists, inter_dists = [], []
rng = np.random.default_rng(42)
sample_idx = rng.choice(N, size=min(500, N), replace=False)
for i in sample_idx:
    s = sector_arr[i]
    same = np.where(sector_arr == s)[0]
    diff = np.where(sector_arr != s)[0]
    if len(same) > 1 and len(diff) > 0:
        same_sample = rng.choice(same[same != i], size=min(20, len(same)-1), replace=False)
        diff_sample = rng.choice(diff, size=min(20, len(diff)), replace=False)
        intra_dists.append(np.linalg.norm(emb_normed[i] - emb_normed[same_sample], axis=1).mean())
        inter_dists.append(np.linalg.norm(emb_normed[i] - emb_normed[diff_sample], axis=1).mean())

intra_mean = np.mean(intra_dists)
inter_mean = np.mean(inter_dists)
sep_ratio  = inter_mean / intra_mean

print("=" * 55)
print("CONCLUSION SECTION 1 — Sanity Check")
print("=" * 55)
print(f"Distance intra-secteur moyenne (L2 cosine) : {intra_mean:.4f}")
print(f"Distance inter-secteur moyenne (L2 cosine) : {inter_mean:.4f}")
print(f"Ratio inter/intra                           : {sep_ratio:.2f}")
print()
if sep_ratio > 1.2:
    print("✓ Les clusters sectoriels sont VISIBLES — le modèle a capturé")
    print("  la structure sectorielle (ratio inter/intra > 1.2).")
else:
    print("⚠ Les clusters sectoriels sont FAIBLEMENT séparés (ratio < 1.2).")
    print("  Vérification visuelle PCA/t-SNE recommandée.")


---
## Section 2 · Calibration

**Paires vraies** (même secteur économique, forte substituabilité attendue) :
`AIR.PA/SAF.PA`, `BNP.PA/GLE.PA`, `MC.PA/KER.PA`, `TTE.PA/ENI.MI`

**Paires fausses** (secteurs différents, faible substituabilité attendue) :
`AIR.PA/EDP.LS`, `SAF.PA/JMT.LS`, `MC.PA/EDP.LS`, `TTE.PA/ABI.BR`

> **Note :** `TTE.PA` (TotalEnergies) est absent de l'univers d'entraînement — les paires impliquant ce ticker seront ignorées.


In [ ]:
PAIRS_TRUE  = [("AIR.PA","SAF.PA"), ("BNP.PA","GLE.PA"), ("MC.PA","KER.PA"), ("TTE.PA","ENI.MI")]
PAIRS_FALSE = [("AIR.PA","EDP.LS"), ("SAF.PA","JMT.LS"), ("MC.PA","EDP.LS"), ("TTE.PA","ABI.BR")]

ticker_set = set(tickers)

def filter_pairs(pairs):
    valid, skipped = [], []
    for a, b in pairs:
        missing = [t for t in (a, b) if t not in ticker_set]
        if missing:
            skipped.append((a, b, missing))
        else:
            valid.append((a, b))
    return valid, skipped

pairs_true_ok,  skip_t = filter_pairs(PAIRS_TRUE)
pairs_false_ok, skip_f = filter_pairs(PAIRS_FALSE)

for a, b, miss in skip_t + skip_f:
    print(f"  ⚠ Paire ignorée {a}/{b} : ticker(s) absent(s) {miss}")

sims_true  = [engine.similarity(a, b) for a, b in pairs_true_ok]
sims_false = [engine.similarity(a, b) for a, b in pairs_false_ok]

rows = []
for (a, b), s in zip(pairs_true_ok, sims_true):
    rows.append({"Paire": f"{a} / {b}", "Label": "VRAIE ✓",
                 "Sim cosine [0,1]": round(s, 4),
                 "Secteur A": engine.sector_lookup.get(a, "?"),
                 "Secteur B": engine.sector_lookup.get(b, "?")})
for (a, b), s in zip(pairs_false_ok, sims_false):
    rows.append({"Paire": f"{a} / {b}", "Label": "FAUSSE ✗",
                 "Sim cosine [0,1]": round(s, 4),
                 "Secteur A": engine.sector_lookup.get(a, "?"),
                 "Secteur B": engine.sector_lookup.get(b, "?")})

df_cal = pd.DataFrame(rows)
print(df_cal.to_string(index=False))
print()
print(f"Vraies  — mean: {np.mean(sims_true):.4f}  min: {min(sims_true):.4f}  max: {max(sims_true):.4f}")
print(f"Fausses — mean: {np.mean(sims_false):.4f}  min: {min(sims_false):.4f}  max: {max(sims_false):.4f}")


In [ ]:
# 2.2 · Histogramme des deux distributions + gap
gap = np.mean(sims_true) - np.mean(sims_false)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Histogram
ax = axes[0]
ax.hist(sims_true,  bins=10, alpha=0.75, color="green", label=f"Vraies (n={len(sims_true)})",  density=False)
ax.hist(sims_false, bins=10, alpha=0.75, color="tomato", label=f"Fausses (n={len(sims_false)})", density=False)
ax.axvline(np.mean(sims_true),  color="darkgreen", ls="--", lw=2, label=f"μ vraie = {np.mean(sims_true):.3f}")
ax.axvline(np.mean(sims_false), color="darkred",   ls="--", lw=2, label=f"μ fausse = {np.mean(sims_false):.3f}")
ax.axvline(0.85, color="black", ls=":", lw=1.5, label="Seuil 0.85")
ax.set_xlabel("Score de similarité cosine [0, 1]")
ax.set_ylabel("Nombre de paires")
ax.set_title(f"Calibration — Gap = {gap:.4f}")
ax.legend(fontsize=9)
ax.set_xlim(0, 1)

# ROC
y_true_roc  = [1]*len(sims_true) + [0]*len(sims_false)
y_score_roc = sims_true + sims_false
fpr, tpr, roc_thresholds = roc_curve(y_true_roc, y_score_roc)
roc_auc = auc(fpr, tpr)

ax2 = axes[1]
ax2.plot(fpr, tpr, "bo-", lw=2, ms=7, label=f"ROC (AUC = {roc_auc:.3f})")
ax2.fill_between(fpr, tpr, alpha=0.12, color="blue")
ax2.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Aléatoire")
ax2.set_xlabel("Taux de faux positifs")
ax2.set_ylabel("Taux de vrais positifs")
ax2.set_title("Courbe ROC — Paires de calibration")
ax2.legend(fontsize=9)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "S2_calibration.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"Gap moyen (cosine [0,1]) : {gap:.4f}")
print(f"AUC ROC                  : {roc_auc:.4f}")


---
## Section 3 · Distribution des similarités

Analyse de la matrice de similarité cosine complète (N×N = 1 256 × 1 256).


In [ ]:
# 3.1 · Histogramme global (100k paires aléatoires)
rng = np.random.default_rng(42)
n_sample = 100_000
i_idx = rng.integers(0, N, n_sample)
j_idx = rng.integers(0, N, n_sample)
mask  = i_idx != j_idx
i_idx, j_idx = i_idx[mask], j_idx[mask]
sample_sims = sim_mat[i_idx, j_idx]

p25, p50, p75, p90, p95 = np.percentile(sample_sims, [25, 50, 75, 90, 95])

fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(sample_sims, bins=80, color="steelblue", edgecolor="white", alpha=0.85)
for p, label, color in [(p50,"p50","orange"), (p90,"p90","red"), (p95,"p95","darkred")]:
    ax.axvline(p, color=color, ls="--", lw=1.5, label=f"{label} = {p:.3f}")
ax.axvline(0.85, color="black", ls=":", lw=2, label="Seuil 0.85")
ax.set_xlabel("Similarité cosine [0, 1]")
ax.set_ylabel("Nombre de paires")
ax.set_title(f"Distribution de 100k paires aléatoires — mean={sample_sims.mean():.3f}  std={sample_sims.std():.3f}")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "S3_sim_hist.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"p25={p25:.3f}  p50={p50:.3f}  p75={p75:.3f}  p90={p90:.3f}  p95={p95:.3f}")


In [ ]:
# 3.2 · Boxplot par secteur (mean similarity de chaque ticker vers tous les autres)
ticker_mean_sim = {tickers[i]: sim_mat[i, [j for j in range(N) if j != i]].mean() for i in range(N)}
sector_sim_data = {s: [ticker_mean_sim[tickers[i]] for i in range(N) if sector_arr[i] == s]
                   for s in unique_sectors}

# Trier par médiane décroissante
sector_order = sorted(unique_sectors, key=lambda s: np.median(sector_sim_data[s]), reverse=True)

fig, ax = plt.subplots(figsize=(14, 7))
bp = ax.boxplot(
    [sector_sim_data[s] for s in sector_order],
    labels=[f"{s}\n(n={len(sector_sim_data[s])})" for s in sector_order],
    patch_artist=True, vert=True, notch=False,
)
colors_bp = sns.color_palette("tab20", len(sector_order))
for patch, c in zip(bp["boxes"], colors_bp):
    patch.set_facecolor(c)
    patch.set_alpha(0.75)
ax.set_ylabel("Similarité cosine moyenne (ticker vs tous les autres)")
ax.set_title("Distribution des similarités moyennes par secteur")
plt.setp(ax.get_xticklabels(), fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "S3_boxplot_sector.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# 3.3 · Heatmap inter-secteurs : similarité cosine moyenne secteur × secteur
sec_idx = {s: np.where(sector_arr == s)[0] for s in unique_sectors}
S = len(unique_sectors)
heatmap_mat = np.zeros((S, S), dtype=np.float32)

for i, si in enumerate(unique_sectors):
    for j, sj in enumerate(unique_sectors):
        idx_i = sec_idx[si]
        idx_j = sec_idx[sj]
        if len(idx_i) == 0 or len(idx_j) == 0:
            heatmap_mat[i, j] = np.nan
            continue
        block = sim_mat[np.ix_(idx_i, idx_j)]
        if i == j:
            # Intra-sector: exclude diagonal
            diag_mask = np.eye(len(idx_i), dtype=bool)
            off_diag = block[~diag_mask]
            heatmap_mat[i, j] = off_diag.mean() if len(off_diag) > 0 else np.nan
        else:
            heatmap_mat[i, j] = block.mean()

df_hm = pd.DataFrame(heatmap_mat, index=unique_sectors, columns=unique_sectors)

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(
    df_hm, ax=ax, annot=True, fmt=".3f", cmap="RdYlGn",
    vmin=0.45, vmax=0.75,
    linewidths=0.5, annot_kws={"size": 8},
)
ax.set_title("Heatmap similarité cosine moyenne inter-secteurs\n(diagonale = intra-secteur hors self-pairs)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha="right", fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "S3_heatmap_sectors.png", dpi=120, bbox_inches="tight")
plt.show()

# Secteurs les plus cohérents intra (forte sim interne)
intra_vals = pd.Series({s: df_hm.loc[s, s] for s in unique_sectors}).sort_values(ascending=False)
print("Secteurs les plus cohérents intra-secteur (mean sim) :")
print(intra_vals.round(4).to_string())


---
## Section 4 · Table de substitution (seuil 0.85)

Analyse de la structure de substituabilité issue de l'engine calibré.


In [ ]:
# 4.1 · Distribution du nombre de substituts par ticker
sub_table = pd.read_parquet(table_path)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
counts = sub_table["n_substitutes"]
ax.hist(counts, bins=range(0, counts.max()+2), color="steelblue", edgecolor="white", alpha=0.85, align="left")
ax.axvline(counts.mean(), color="red", ls="--", lw=2, label=f"Moyenne = {counts.mean():.1f}")
ax.axvline(counts.median(), color="orange", ls=":", lw=2, label=f"Médiane = {counts.median():.0f}")
ax.set_xlabel("Nombre de substituts")
ax.set_ylabel("Nombre de tickers")
ax.set_title(f"Distribution des substituts par ticker (seuil 0.85)\n{N} tickers · {(counts==0).sum()} sans substitut")
ax.legend()

ax2 = axes[1]
by_sector = sub_table.copy()
by_sector["sector"] = [engine.sector_lookup.get(t, "Unknown") for t in by_sector["ticker"]]
sector_counts = by_sector.groupby("sector")["n_substitutes"].mean().sort_values(ascending=False)
bars = ax2.barh(sector_counts.index, sector_counts.values, color=sns.color_palette("tab20", len(sector_counts)))
ax2.set_xlabel("Nombre moyen de substituts")
ax2.set_title("Nombre moyen de substituts par secteur")
ax2.axvline(counts.mean(), color="red", ls="--", lw=1.5, label=f"Moy. globale = {counts.mean():.1f}")
ax2.legend(fontsize=9)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "S4_substitutes_dist.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# 4.2 · Tickers isolés (0 substituts) : analyse
isolated = sub_table[sub_table["n_substitutes"] == 0].copy()
isolated["sector"] = [engine.sector_lookup.get(t, "Unknown") for t in isolated["ticker"]]

print(f"Tickers isolés : {len(isolated)} / {N} ({100*len(isolated)/N:.1f} %)")
print()
print(isolated[["ticker","sector"]].sort_values("sector").to_string(index=False))
print()
print("Secteurs les plus représentés parmi les isolés :")
print(isolated["sector"].value_counts().to_string())


In [ ]:
# 4.3 · Top 30 paires les plus similaires
long_table = pd.read_parquet(long_path)
top30 = long_table.drop_duplicates(
    subset=["similarity"]
).nlargest(30, "similarity")[["ticker","substitute","similarity","ticker_sector","sub_sector"]]
top30.index = range(1, len(top30)+1)
print("Top 30 paires les plus similaires (seuil 0.85) :")
print(top30.to_string())


In [ ]:
# 4.4 · Taille des composantes connexes (groupes de substituabilité)
groups = engine.substitution_groups(threshold=0.85)
group_sizes = sorted([len(g) for g in groups], reverse=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.hist(group_sizes, bins=range(2, max(group_sizes)+2), color="mediumpurple",
        edgecolor="white", alpha=0.85, align="left")
ax.set_xlabel("Taille du groupe")
ax.set_ylabel("Nombre de groupes")
ax.set_title(f"Taille des composantes connexes (seuil 0.85)\n{len(groups)} groupes · {sum(group_sizes)} tickers couverts")
ax.set_yscale("log")

ax2 = axes[1]
sizes_arr = np.array(group_sizes)
cum_tickers = np.cumsum(sizes_arr)
ax2.plot(range(1, len(sizes_arr)+1), cum_tickers, "o-", color="mediumpurple", ms=3)
ax2.set_xlabel("Rang du groupe (trié par taille décrois.)")
ax2.set_ylabel("Tickers couverts (cumulé)")
ax2.set_title("Couverture cumulée des groupes de substitution")
ax2.axhline(N, color="red", ls="--", lw=1.5, label=f"Univers total ({N})")
ax2.legend()

fig.tight_layout()
fig.savefig(FIGURES_DIR / "S4_groups.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"Groupes : {len(groups)}  |  Tickers couverts : {sum(group_sizes)}  |  "
      f"Plus grand groupe : {max(group_sizes)}  |  Médiane : {np.median(group_sizes):.0f}")


In [ ]:
# 4.5 · Secteurs les plus substituables vs les plus isolés
by_sec = sub_table.copy()
by_sec["sector"] = [engine.sector_lookup.get(t, "Unknown") for t in by_sec["ticker"]]
sec_stats = by_sec.groupby("sector").agg(
    n_tickers=("ticker","count"),
    mean_subs=("n_substitutes","mean"),
    pct_isolated=("n_substitutes", lambda x: (x==0).mean()*100),
).round(2).sort_values("mean_subs", ascending=False)

print("Secteurs les plus substituables (mean_subs décroissant) :")
print(sec_stats.to_string())


---
## Section 5 · Cas d'usage `compare_trades`

`compare_trades(trader_x, trader_y)` effectue un matching greedy :
- **overlap_strict** : mêmes tickers
- **overlap_substitutable** : tickers différents mais substituables (sim ≥ seuil)
- **substitutable_score** : fraction du portefeuille X expliquée par Y

Les valeurs du dict correspondent aux tailles de position (ici normalisées à 1.0).


In [ ]:
# Ex1 · Trader X : AIR.PA  |  Trader Y : SAF.PA
# Airbus vs Safran — deux leaders de l'aéronautique europenne
ex1 = engine.compare_trades({"AIR.PA": 1.0}, {"SAF.PA": 1.0})
sim_ex1 = engine.similarity("AIR.PA","SAF.PA")
print("=" * 60)
print("EXEMPLE 1 : AIR.PA (Airbus) vs SAF.PA (Safran)")
print(f"Secteur AIR.PA : {engine.sector_lookup.get('AIR.PA','?')}")
print(f"Secteur SAF.PA : {engine.sector_lookup.get('SAF.PA','?')}")
print(f"Similarité cosine : {sim_ex1:.4f}")
print(f"Résultat compare_trades :")
for k, v in ex1.items():
    print(f"  {k}: {v}")
print()
score = ex1["substitutable_score"]
if score >= 0.85:
    verdict = "DÉCISIONS ÉQUIVALENTES — les deux ordres visent la même exposition"
elif score >= 0.50:
    verdict = "PARTIELLEMENT ÉQUIVALENTES"
else:
    verdict = "PAS ÉQUIVALENTES — expositions distinctes"
print(f"Interprétation métier : {verdict}  (score={score:.4f})")


In [ ]:
# Ex2 · Trader X : MC.PA + TTE.PA (si présent)  |  Trader Y : KER.PA + ENI.MI
# LVMH+TotalEnergies vs Kering+ENI
# TTE.PA absent de l'univers → géré silencieusement (contribution ignorée dans total_x)
x_portfolio = {"MC.PA": 1.0}
if "TTE.PA" in ticker_set:
    x_portfolio["TTE.PA"] = 1.0
else:
    print("⚠ TTE.PA absent de l'univers — portefeuille X réduit à MC.PA")

ex2 = engine.compare_trades(x_portfolio, {"KER.PA": 1.0, "ENI.MI": 1.0})
sim_mc_ker = engine.similarity("MC.PA","KER.PA")
print("=" * 60)
print("EXEMPLE 2 : MC.PA (+TTE.PA*) vs KER.PA + ENI.MI")
print(f"Similarité MC.PA/KER.PA : {sim_mc_ker:.4f}")
print(f"Résultat compare_trades :")
for k, v in ex2.items():
    print(f"  {k}: {v}")
print()
score2 = ex2["substitutable_score"]
if score2 >= 0.85:
    verdict2 = "DÉCISIONS ÉQUIVALENTES — même exposition luxe (+ énergie si TTE.PA présent)"
elif score2 >= 0.50:
    verdict2 = "PARTIELLEMENT ÉQUIVALENTES — seule la jambe luxe est matchée"
else:
    verdict2 = "PAS ÉQUIVALENTES"
print(f"Interprétation métier : {verdict2}  (score={score2:.4f})")
print("Note : LVMH et Kering sont les deux géants mondiaux du luxe — forte co-substituabilité attendue.")


In [ ]:
# Ex3 · Paire cross-secteur surprenante
# GNRO.PA (Healthcare) ↔ ZENA.OL (Energy) : similarité 0.9961
# Deux small-caps de marchés différents présentant un profil de prix quasi-identique
ex3 = engine.compare_trades({"GNRO.PA": 1.0}, {"ZENA.OL": 1.0})
sim_ex3 = engine.similarity("GNRO.PA", "ZENA.OL")
sec_gnro = engine.sector_lookup.get("GNRO.PA","?")
sec_zena = engine.sector_lookup.get("ZENA.OL","?")

print("=" * 60)
print("EXEMPLE 3 : GNRO.PA vs ZENA.OL — paire cross-secteur surprenante")
print(f"GNRO.PA secteur : {sec_gnro}")
print(f"ZENA.OL secteur : {sec_zena}")
print(f"Similarité cosine : {sim_ex3:.4f}  (top cross-secteur de tout l'univers !)")
print(f"Résultat compare_trades :")
for k, v in ex3.items():
    print(f"  {k}: {v}")
print()
print("Interprétation métier :")
print("  Ces deux valeurs — Guerbet (médecine de l'image, Paris) et Zena Group (énergie,")
print("  Oslo) — affichent des profils de prix quasi-identiques sur 2024 malgré l'absence")
print("  de lien sectoriel direct. Le modèle capte une dynamique de marché commune")
print("  (micro/small-cap, bêta similaire, corrélation de flux) plutôt qu'une réalité")
print("  fondamentale. C'est un signal à interpréter avec PRUDENCE : cette substituabilité")
print("  est technique, pas économique. Le filtre same_sector_only=True l'éliminera.")


---
## Section 6 · Optimisation du seuil

Precision / Recall / F1 calculés sur les paires de calibration disponibles (excluant TTE.PA).


In [ ]:
# 6.1 · Sweep des seuils 0.70 → 0.98 (pas 0.01)
thresholds = np.arange(0.70, 0.99, 0.01)
y_true_arr  = np.array([1]*len(sims_true) + [0]*len(sims_false))
y_score_arr = np.array(sims_true + sims_false)

results = []
for t in thresholds:
    pred = (y_score_arr >= t).astype(int)
    tp = int(((pred == 1) & (y_true_arr == 1)).sum())
    fp = int(((pred == 1) & (y_true_arr == 0)).sum())
    fn = int(((pred == 0) & (y_true_arr == 1)).sum())
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2*prec*rec / (prec + rec) if (prec + rec) > 0 else 0.0
    results.append({"threshold": round(t, 2), "precision": prec,
                    "recall": rec, "f1": f1, "tp": tp, "fp": fp, "fn": fn})

df_thresh = pd.DataFrame(results)
best_row   = df_thresh.loc[df_thresh["f1"].idxmax()]
opt_thresh = best_row["threshold"]

print(df_thresh[["threshold","precision","recall","f1"]].to_string(index=False))
print(f"\nSeuil optimal F1 : {opt_thresh:.2f}  (F1 = {best_row['f1']:.4f})")


In [ ]:
# 6.2 · Courbe F1 vs seuil
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(df_thresh["threshold"], df_thresh["f1"],       "b-o", ms=5, lw=2, label="F1")
ax.plot(df_thresh["threshold"], df_thresh["precision"], "g--s", ms=4, lw=1.5, label="Precision")
ax.plot(df_thresh["threshold"], df_thresh["recall"],    "r--^", ms=4, lw=1.5, label="Recall")

ax.axvline(opt_thresh, color="blue", ls=":", lw=2,
           label=f"Seuil optimal = {opt_thresh:.2f}  (F1={best_row['f1']:.3f})")
ax.axvline(0.85, color="black", ls="--", lw=1.5, label="Seuil README = 0.85")

ax.set_xlabel("Seuil de similarité")
ax.set_ylabel("Score")
ax.set_title("Precision / Recall / F1 vs seuil de similarité")
ax.legend(fontsize=9)
ax.set_ylim(-0.05, 1.10)
ax.set_xlim(0.69, 0.99)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "S6_threshold_optim.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"Seuil optimal : {opt_thresh:.2f}  vs README : 0.85")
delta = opt_thresh - 0.85
print(f"Écart : {'+' if delta>=0 else ''}{delta:.2f}  "
      f"({'optimal > README' if delta > 0 else 'optimal < README' if delta < 0 else 'identique'})")


---
## Section 7 · Résumé Go/No-Go

Cellule finale récapitulative — s'affiche automatiquement à chaque run.

| Critère | Cible | Valeur mesurée | Statut |
|---|---|---|---|
| Gap calibration | > 0.50 | (calculé ci-dessous) | (auto) |
| AUC ROC | > 0.85 | (calculé ci-dessous) | (auto) |
| Clusters PCA/t-SNE | visibles | *vérification visuelle* | ← manuel |
| Seuil optimal | — | (calculé ci-dessous) | — |


In [ ]:
TARGET_GAP = 0.50
TARGET_AUC = 0.85

gap_ok = gap >= TARGET_GAP
auc_ok = roc_auc >= TARGET_AUC

def badge(ok):
    return "✓ OK" if ok else "✗ FAIL"

print("=" * 65)
print(f"  RÉSUMÉ GO/NO-GO — {SAVE_NAME}  ({N} tickers, ref 2024)")
print("=" * 65)
print(f"  Gap calibration       : {gap:.4f}  (cible > {TARGET_GAP})  → {badge(gap_ok)}")
print(f"  AUC ROC               : {roc_auc:.4f}  (cible > {TARGET_AUC})  → {badge(auc_ok)}")
print(f"  Clusters sectoriels   : vérification visuelle PCA/t-SNE (section 1)")
print(f"  Seuil optimal (F1)    : {opt_thresh:.2f}  (README : 0.85)")
print(f"  Tickers couverts ≥1   : {N - (sub_table['n_substitutes']==0).sum()}/{N}  "
      f"({100*(1-(sub_table['n_substitutes']==0).mean()):.1f} %)")
print(f"  Tickers isolés        : {(sub_table['n_substitutes']==0).sum()}  "
      f"({100*(sub_table['n_substitutes']==0).mean():.1f} %)")
print()
print("-" * 65)

all_auto_ok = gap_ok and auc_ok
if all_auto_ok:
    print("  VERDICT (critères automatiques) : ✓ PRÊT PRODUCTION")
    print("  → Vérifier visuellement les clusters PCA/t-SNE avant déploiement.")
else:
    fails = []
    if not gap_ok:
        fails.append(f"gap={gap:.4f} < {TARGET_GAP}")
    if not auc_ok:
        fails.append(f"AUC={roc_auc:.4f} < {TARGET_AUC}")
    print(f"  VERDICT : AMÉLIORATION NÉCESSAIRE")
    print(f"  → Critères non atteints : {', '.join(fails)}")

print("=" * 65)
